In [ ]:
import pandas as pd

agenda = pd.read_csv("data/raw/openagenda_api.csv", sep=";")

print(agenda.shape)

(11756, 56)


In [ ]:
agenda = agenda.rename(columns={
    "uid": "Identifiant",
    "title_fr": "Titre",
    "description_fr": "Description",
    "longdescription_fr": "Description longue",
    "keywords_fr": "Mots clés",
    "firstdate_begin": "Première date - Début",
    "lastdate_end": "Dernière date - Fin",
    "location_name": "Nom du lieu",
    "location_address": "Adresse",
    "location_city": "Ville",
    "conditions_fr": "Détail des conditions",
    "canonicalurl": "URL canonique"
})

In [ ]:
colonnes = [
    "Identifiant", "Titre", "Description",
    "Description longue", "Mots clés",
    "Première date - Début", "Dernière date - Fin",
    "Nom du lieu", "Adresse", "Ville",
    "Détail des conditions", "URL canonique"
]

agenda_clean = agenda[colonnes].copy()

Conversion des dates

In [ ]:
for col in ["Première date - Début", "Dernière date - Fin"]:
    agenda_clean[col] = pd.to_datetime(
        agenda_clean[col],
        format="mixed",
        errors="coerce",
        utc=True
    )

Nettoyage des descriptions et et des évènements

In [ ]:
from html import unescape

date_limite = pd.Timestamp.now(tz="UTC") - pd.DateOffset(years=1)

agenda_clean = agenda_clean[
    (agenda_clean["Dernière date - Fin"] >= date_limite) &
    (agenda_clean["Ville"].str.strip().str.lower() == "paris")
].copy()

agenda_clean = agenda_clean.drop_duplicates(subset="Identifiant")
agenda_clean = agenda_clean.dropna(subset=["Titre"])

colonnes_texte = [
    "Description", "Description longue",
    "Mots clés", "Nom du lieu",
    "Adresse", "Détail des conditions"
]

agenda_clean[colonnes_texte] = agenda_clean[colonnes_texte].fillna("")

for col in ["Titre", "Description", "Description longue"]:
    agenda_clean[col] = (
        agenda_clean[col]
        .str.replace(r"<[^>]+>", " ", regex=True)
        .apply(unescape)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

new_agenda_clean = agenda_clean[
    (agenda_clean["Description"] != "") |
    (agenda_clean["Description longue"] != "")
].copy()

print("Événements après nettoyage :", len(new_agenda_clean))

Événements après nettoyage : 11718


Création de la colonne texte rag

In [ ]:
new_agenda_clean["texte_rag"] = (
    "Événement : " + new_agenda_clean["Titre"] +
    ". Description : " + new_agenda_clean["Description"] +
    ". Informations : " + new_agenda_clean["Description longue"] +
    ". Lieu : " + new_agenda_clean["Nom du lieu"] +
    ". Adresse : " + new_agenda_clean["Adresse"] +
    ". Début : " + new_agenda_clean["Première date - Début"].astype(str) +
    ". Fin : " + new_agenda_clean["Dernière date - Fin"].astype(str) +
    ". Conditions : " + new_agenda_clean["Détail des conditions"]
)

new_agenda_clean.to_csv(
    "data/processed/openagenda_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dimensions finales :", new_agenda_clean.shape)

Dimensions finales : (11718, 13)


Vérification des résultats

In [37]:
print(df_clean["Ville"].value_counts())
print(df_clean["Dernière date - Fin"].min())
print(df_clean["Identifiant"].duplicated().sum())

Ville
Paris    11718
Name: count, dtype: int64
2025-10-27 16:30:00+00:00
0


In [16]:
agenda_df["Identifiant"].duplicated().sum()

np.int64(0)

In [17]:
colonnes = [
    "Identifiant",
    "Titre",
    "Description",
    "Description longue",
    "Mots clés",
    "Première date - Début",
    "Dernière date - Fin",
    "Nom du lieu",
    "Adresse",
    "Détail des conditions",
    "URL canonique"
]

df_clean = df[colonnes].copy()

df_clean.head()

,Identifiant,Titre,Description,Description longue,Mots clés,Première date - Début,Dernière date - Fin,Nom du lieu,Adresse,Détail des conditions,URL canonique
0,99169752,NaN,NaN,NaN,NaN,2026-02-05T19:00:00+01:00,2026-02-05T21:00:00+01:00,HEC Alumni,9 avenue Franklin Roosevelt 75008 Paris,NaN,https://openagenda.com/unidivers-oui-sortir/ev...
1,73472317,Visite commentée - Sur inscription uniquement,Visite commentée,<p>Le regard de la Lanterne a été parachevé en...,NaN,2025-09-20T12:00:00+02:00,2025-09-21T16:30:00+02:00,Regard de la Lanterne,213 rue de Belleville 75019 Paris,Inscription à partir du samedi 5 septembre,https://openagenda.com/culture/events/visita-c...
2,15115519,Cabaret d'Elsa #2,A suivre...,<p>A suivre...</p>,NaN,2026-12-17T19:00:00+01:00,2026-12-17T21:00:00+01:00,Le Zèbre de Belleville,"63 Boulevard de Belleville, 75011 Paris, France",NaN,https://openagenda.com/saison-cabaret2026/even...
3,6141408,Eruptions de magnétars et feux d'artifice élec...,Soirée astronomie avec la participation de Jen...,"<p>Connaissez-vous les magnétars, ces étoiles ...",NaN,2026-09-22T18:30:00+02:00,2026-09-22T21:30:00+02:00,Les Étincelles du Palais de la découverte,"186 Rue St Charles, 75015 Paris","5,50 € - Tarif mobilité durable : 4€",https://openagenda.com/echosciences/events/eru...
4,64075534,14H REAL,Présentation,"<p>De la Comédie-Française, De Martin Darondea...","Les Cinq Caumartin,14H REAL",2026-07-22T13:45:00+02:00,2026-07-22T16:45:00+02:00,Maison de la culture du Japon à Paris,"101 bis Quai Jacques Chirac, Paris 15e Arrondi...",NaN,https://openagenda.com/scare/events/14h-real-7...


Filtrer par date

In [19]:
df_clean["Première date - Début"].head(10)

0    2026-02-05T19:00:00+01:00
1    2025-09-20T12:00:00+02:00
2    2026-12-17T19:00:00+01:00
3    2026-09-22T18:30:00+02:00
4    2026-07-22T13:45:00+02:00
5    2026-03-28T11:00:00+01:00
6    2025-09-20T14:30:00+02:00
7    2025-09-20T10:00:00+02:00
8    2025-09-20T10:00:00+02:00
9    2025-09-20T09:00:00+02:00
Name: Première date - Début, dtype: str

In [20]:
df_clean["Première date - Début"] = pd.to_datetime(
    df_clean["Première date - Début"],
    format="mixed",
    errors="coerce",
    utc=True
)

df_clean["Dernière date - Fin"] = pd.to_datetime(
    df_clean["Dernière date - Fin"],
    format="mixed",
    errors="coerce",
    utc=True
)

In [21]:
df_clean[["Première date - Début", "Dernière date - Fin"]].head()

,Première date - Début,Dernière date - Fin
0,2026-02-05 18:00:00+00:00,2026-02-05 20:00:00+00:00
1,2025-09-20 10:00:00+00:00,2025-09-21 14:30:00+00:00
2,2026-12-17 18:00:00+00:00,2026-12-17 20:00:00+00:00
3,2026-09-22 16:30:00+00:00,2026-09-22 19:30:00+00:00
4,2026-07-22 11:45:00+00:00,2026-07-22 14:45:00+00:00


In [22]:
df_clean[["Première date - Début", "Dernière date - Fin"]].isna().sum()

Première date - Début    0
Dernière date - Fin      0
dtype: int64

In [23]:
date_limite = pd.Timestamp.now(tz="UTC") - pd.DateOffset(years=1)

df_clean = df_clean[
    df_clean["Dernière date - Fin"] >= date_limite
]

print("Événements conservés :", len(df_clean))

Événements conservés : 10652


Ici nous sommes passés de 15312 données à 10 652 sur le nouveau df après avoir trier les dates pour garder les données les plus récentes < 1 an

Supprimer les doublons sur le nouveau DF 

In [24]:
print("Doublons :", df_clean["Identifiant"].duplicated().sum())

df_clean = df_clean.drop_duplicates(subset="Identifiant").copy()

print("Événements restants :", len(df_clean))

Doublons : 0
Événements restants : 10652


Conserver les évènements ayant au moins un titre et une description

In [25]:
df_clean = df_clean.dropna(subset=["Titre"])

colonnes_texte = [
    "Description",
    "Description longue",
    "Mots clés",
    "Nom du lieu",
    "Adresse",
    "Détail des conditions"
]

df_clean[colonnes_texte] = df_clean[colonnes_texte].fillna("")

df_clean = df_clean[
    (df_clean["Description"].str.strip() != "") |
    (df_clean["Description longue"].str.strip() != "")
].copy()

print("Événements restants :", len(df_clean))

Événements restants : 10631


Nettoyer les descriptions avec du HTML

In [26]:
from html import unescape

for col in ["Titre", "Description", "Description longue"]:
    df_clean[col] = (
        df_clean[col]
        .str.replace(r"<[^>]+>", " ", regex=True)
        .apply(unescape)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

Vérification de la zone géographique; J'ai choisi Paris comme zone d'activité

In [ ]:
[col for col in df.columns if "ville" in col.lower()]

df_clean["Ville"] = df.loc[df_clean.index, "Ville"]

df_clean = df_clean[
    df_clean["Ville"].str.strip().str.lower() == "paris"
].copy()

print("Événements à Paris :", len(df_clean))




Événements à Paris : 10631


preparer les données destinées au Rag

In [29]:
df_clean["texte_rag"] = (
    "Événement : " + df_clean["Titre"] +
    ". Description : " + df_clean["Description"] +
    ". Informations : " + df_clean["Description longue"] +
    ". Lieu : " + df_clean["Nom du lieu"] +
    ". Adresse : " + df_clean["Adresse"] +
    ". Début : " + df_clean["Première date - Début"].astype(str) +
    ". Fin : " + df_clean["Dernière date - Fin"].astype(str) +
    ". Conditions : " + df_clean["Détail des conditions"]
)

df_clean[["Titre", "texte_rag"]].head()

,Titre,texte_rag
2,Cabaret d'Elsa #2,Événement : Cabaret d'Elsa #2. Description : A...
3,Eruptions de magnétars et feux d'artifice élec...,Événement : Eruptions de magnétars et feux d'a...
4,14H REAL,Événement : 14H REAL. Description : Présentati...
5,Marché européen des artisans moldaves en France,Événement : Marché européen des artisans molda...
30,Initiation au dessin vectoriel avec Inkscape,Événement : Initiation au dessin vectoriel ave...


Cette colonne sera transformée en vecteurs numériques et sera indexée dans FAISS

Exportation des données nettoyées

In [30]:
df_clean.to_csv(
    "data/processed/openagenda_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dimensions finales :", df_clean.shape)

Dimensions finales : (10631, 13)
